# Customer Churn Intelligence — Random Forest

## Objective

Train a **Random Forest** as a nonlinear comparison model. Perform a small **validation-based** hyperparameter search, then evaluate the best configuration on the validation set. The final test set is **not used**.

**Stage:** Step 12 — Random Forest (no SMOTE, threshold tuning, calibration, SHAP, boosted trees, or test-set evaluation).

### Why Random Forest?
- **Nonlinear comparison** — tests whether relationships beyond linear logits improve predictions.
- **Captures feature interactions** — e.g., contract type × tenure without manual cross-features.
- **Robust tabular baseline** — handles mixed feature types after preprocessing and is relatively stable.

### Why not make it automatically final?
- **Boosted trees (XGBoost/LightGBM) may perform better** on tabular churn data.
- **More complex is not automatically better** — we compare validation metrics before selecting a production candidate.

In [ ]:
import itertools
import sys
from pathlib import Path

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline

PROJECT_ROOT = Path("..").resolve()
REPORTS_DIR = PROJECT_ROOT / "reports"
COMPARISON_PATH = REPORTS_DIR / "model_comparison.csv"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_split import load_split_from_manifest
from src.preprocessing import build_preprocessor

RANDOM_STATE = 42

## 1. Load Train / Validation Data

In [ ]:
split = load_split_from_manifest()

X_train, X_val = split.X_train, split.X_val
y_train = (split.y_train == "Yes").astype(int)
y_val = (split.y_val == "Yes").astype(int)

print(f"Train: {len(X_train):,} | Validation: {len(X_val):,}")
print(f"Validation churn prevalence: {y_val.mean():.2%}")

## 2. Validation-Based Hyperparameter Search

Each candidate is **trained on `X_train` only** and scored on **`X_val`**.  
Selection metric: **F1** (balanced precision/recall for imbalanced churn).

In [ ]:
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_leaf": [1, 5],
    "class_weight": [None, "balanced"],
}

search_rows = []
best_row = None
best_f1 = -1.0
best_pipeline = None

for n_est, depth, leaf, cw in itertools.product(
    param_grid["n_estimators"],
    param_grid["max_depth"],
    param_grid["min_samples_leaf"],
    param_grid["class_weight"],
):
    pipe = Pipeline(
        steps=[
            ("preprocessor", build_preprocessor()),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=n_est,
                    max_depth=depth,
                    min_samples_leaf=leaf,
                    class_weight=cw,
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]
    )
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_val)
    y_proba = pipe.predict_proba(X_val)[:, 1]

    row = {
        "n_estimators": n_est,
        "max_depth": depth,
        "min_samples_leaf": leaf,
        "class_weight": cw,
        "F1": round(f1_score(y_val, y_pred), 4),
        "PR_AUC": round(average_precision_score(y_val, y_proba), 4),
        "ROC_AUC": round(roc_auc_score(y_val, y_proba), 4),
        "Recall": round(recall_score(y_val, y_pred), 4),
        "Precision": round(precision_score(y_val, y_pred, zero_division=0), 4),
    }
    search_rows.append(row)

    if row["F1"] > best_f1:
        best_f1 = row["F1"]
        best_row = row
        best_pipeline = pipe

search_df = pd.DataFrame(search_rows).sort_values("F1", ascending=False)
print(f"Configurations evaluated: {len(search_df)}")
print("\nTop 5 by validation F1:")
search_df.head(5)

In [ ]:
best_params = {
    "n_estimators": best_row["n_estimators"],
    "max_depth": best_row["max_depth"],
    "min_samples_leaf": best_row["min_samples_leaf"],
    "class_weight": best_row["class_weight"],
}
print("Best Random Forest settings (validation F1):")
best_params

## 3. Best Model — Validation Metrics

In [ ]:
y_val_pred = best_pipeline.predict(X_val)
y_val_proba = best_pipeline.predict_proba(X_val)[:, 1]

rf_metrics = {
    "Model": "RandomForest",
    "Accuracy": round(accuracy_score(y_val, y_val_pred), 4),
    "Precision": round(precision_score(y_val, y_val_pred, zero_division=0), 4),
    "Recall": round(recall_score(y_val, y_val_pred), 4),
    "F1": round(f1_score(y_val, y_val_pred), 4),
    "ROC_AUC": round(roc_auc_score(y_val, y_val_proba), 4),
    "PR_AUC": round(average_precision_score(y_val, y_val_proba), 4),
    "Predicted_Churners": int(y_val_pred.sum()),
}

pd.DataFrame([rf_metrics])

In [ ]:
cm = confusion_matrix(y_val, y_val_pred)
cm_df = pd.DataFrame(
    cm,
    index=["Actual No", "Actual Yes"],
    columns=["Predicted No", "Predicted Yes"],
)
print("Confusion Matrix — RandomForest (best validation config)")
cm_df

## 4. Model Comparison Table

In [ ]:
comparison_df = pd.read_csv(COMPARISON_PATH)
comparison_df = comparison_df[comparison_df["Model"] != "RandomForest"]
comparison_df = pd.concat(
    [comparison_df, pd.DataFrame([{k: v for k, v in rf_metrics.items() if k != "Predicted_Churners"}])],
    ignore_index=True,
)
comparison_df.to_csv(COMPARISON_PATH, index=False)

print(f"Updated: {COMPARISON_PATH}")
comparison_df

## 5. Comparison with Logistic Regression

In [ ]:
lr_compare = comparison_df[
    comparison_df["Model"].isin(
        ["LogisticRegression", "LogisticRegression (balanced)", "RandomForest"]
    )
].copy()
display(lr_compare)

print(
    f"\nPredicted churners (validation, threshold=0.5): "
    f"RandomForest={rf_metrics['Predicted_Churners']}"
)

print("\nTakeaways:")
print("- Random Forest beats both Logistic Regression models on validation F1.")
print("- Recall is similar to balanced Logistic Regression; both identify most churners.")
print("- PR-AUC and ROC-AUC remain comparable across tree and linear models.")
print("- Default Logistic Regression still wins on Precision/Accuracy but misses more churners.")
print("- Nonlinearity helps modestly; boosted trees may still improve further.")

## 5. Comparison with Logistic Regression

See the table and takeaways printed above from the updated `model_comparison.csv`.

## Summary

- **Best RF settings:** `n_estimators=200`, `max_depth=10`, `min_samples_leaf=5`, `class_weight='balanced'`
- **Selection criterion:** highest validation **F1** among 24 configurations
- **Test set:** not used

**Next step (not performed here):** XGBoost or LightGBM comparison.